## Goal: Filter desired rows from a pandas DataFrame using conditions.

### Key Concepts:
* **Boolean condition**
* **Boolean Series**
* **Row filtering**
* **Comparison operators**
* **Multiple conditions**
* **`&` (AND) and `|` (OR)**
* **`isin()`**
* **`between()`**
* **String filtering**
* **`loc[]`**

---
## 01. Row Filtering
**Row filtering** means selecting only the rows from a DataFrame that meet specific conditions.

For example, let's say we have the following student data:

```text
Name    Age   Score   Passed
Alice   25    88      True
Bob     30    92      True
Charlie 35    79      False 
David   28    85      True
```

If we want to see only the students whose score is 85 or higher:

&nbsp;&nbsp;&nbsp;&nbsp;df_students[df_students["Score"] >= 85]

**Result:**

```text
Name    Age   Score   Passed
Alice   25     88     True
Bob     30     92     True
David   28     85     True

---
## 02. Basic Principle: Boolean Condition

Pandas filtering works by first creating a condition, and then retrieving only the rows where that condition is `True`.

```python
df_students["Score"] >= 85
````
This code returns True or False for each row.

For example:
```text
0     True
1     True
2    False
3     True
Name: Score, dtype: bool
```
You can think of this as a Boolean Series.

When you pass this Boolean Series into the DataFrame:

```Python
df_students[df_students["Score"] >= 85]
```
Only the rows that are True will be selected.

---
## 03. Creating a Practice CSV File

In [77]:
# Import modules
import os
import pandas as pd

path_dir = "../assets/Week05/Day04/Input/"
file_name = "students.csv"
full_file_path = f"{path_dir}{file_name}"

# Create input folder if it doesn't exist
os.makedirs(path_dir, exist_ok=True)

# Create sample student data
student_data = {
    "Name": ["Alice", "Bob", "Charlie", "David", "Eva", "Frank"],
    "Age": [25, 30, 35, 28, 22, 40],
    "Score": [88, 92, 79, 85, 67, 95],
    "Department": ["Data", "Engineering", "Data", "Cloud", "Engineering", "Security"],
    "Passed": [True, True, False, True, False, True]
}

# Create a DataFrame
df_students = pd.DataFrame(student_data)

# Save the DataFrame as a CSV file
df_students.to_csv(full_file_path, index=False)

print("students.csv has been created.")

students.csv has been created.


---
## 04. Reading the CSV file created above

In [4]:
# Read the students CSV file
df_students = pd.read_csv(full_file_path)

# Display the DataFrame
df_students

,Name,Age,Score,Department,Passed
0,Alice,25,88,Data,True
1,Bob,30,92,Engineering,True
2,Charlie,35,79,Data,False
3,David,28,85,Cloud,True
4,Eva,22,67,Engineering,False
5,Frank,40,95,Security,True


## 05. Row Filtering with a Single Condition
e.g.> Select rows where the `score` is 85 or higher.

In [5]:
# Filter rows where Score is greater than or equal to 85
high_score_students = df_students[df_students["Score"] >= 85]

high_score_students

,Name,Age,Score,Department,Passed
0,Alice,25,88,Data,True
1,Bob,30,92,Engineering,True
3,David,28,85,Cloud,True
5,Frank,40,95,Security,True


e.g.> Select rows where the `Age` is 30 or higher

In [6]:
age_30_or_higher = df_students[df_students["Age"]>=30]
age_30_or_higher

,Name,Age,Score,Department,Passed
1,Bob,30,92,Engineering,True
2,Charlie,35,79,Data,False
5,Frank,40,95,Security,True


String conditions are also possible (==):

e.g.> Select rows where the `Department` is 'Data'.

In [7]:
data_department = df_students[df_students["Department"]=="Data"]
data_department

,Name,Age,Score,Department,Passed
0,Alice,25,88,Data,True
2,Charlie,35,79,Data,False


## 06. Filtering by a Boolean Column
- If a column already contains Boolean values (`True` or `False`), you can filter the DataFrame to select only the rows that are `True` by passing the column directly.

In [12]:
# Filter rows using the boolean column directly
passed_students = df_students[df_students["Passed"]]

passed_students

,Name,Age,Score,Department,Passed
0,Alice,25,88,Data,True
1,Bob,30,92,Engineering,True
3,David,28,85,Cloud,True
5,Frank,40,95,Security,True


## 07. Filtering with Multiple Conditions

When using multiple conditions in pandas, you must use bitwise operators instead of Python's logical keywords (`and`, `or`, `not`):

- `&` → **AND** (Both conditions must be True)
- `|` → **OR** (At least one condition must be True)
- `~` → **NOT** (Inverts the condition)

⚠️ **CRITICAL RULE:** Each condition **must** be wrapped in parentheses `( )`. Leaving them out will cause a `ValueError`.

- Syntax Structure
```python
# Correct Style:
df[(Condition1) & (Condition2)]

# Wrong Style (Will raise an error):
df[Condition1 and Condition2]  # ❌ Bug: Cannot use 'and'
df[df["Age"] >= 20 & df["Score"] >= 85]  # ❌ Bug: Missing parentheses
```

- Why are parentheses `( )` mandatory?<br>
It is because of **Python's operator precedence**. 
If you don't use parentheses, pandas evaluates the bitwise operator (`&`) before the comparison operator (`>=`). In other words, Python tries to calculate `20 & df["Score"]` first, which leads to an unexpected `ValueError`. 
To prevent this incorrect order of operations, you must wrap each condition in parentheses.

In [19]:
# e.g.1> and (&)

# Filter rows using two conditions with &
filtered_students = df_students[
    (df_students["Score"] >= 85) & (df_students["Age"] >= 30)
]

filtered_students

,Name,Age,Score,Department,Passed
1,Bob,30,92,Engineering,True
5,Frank,40,95,Security,True


In [20]:
# e.g.2> or (|)

# Filter rows using OR condition with |
data_or_cloud = df_students[
    (df_students["Department"] == "Data") | (df_students["Department"] == "Cloud")
]

data_or_cloud

,Name,Age,Score,Department,Passed
0,Alice,25,88,Data,True
2,Charlie,35,79,Data,False
3,David,28,85,Cloud,True


In [21]:
# e.g.3> not (~)

# Filter rows where Passed is not True
not_passed_students = df_students[~df_students["Passed"]]

not_passed_students

,Name,Age,Score,Department,Passed
2,Charlie,35,79,Data,False
4,Eva,22,67,Engineering,False


## 08. Filtering with Multiple Allowed Values: `isin()`
When you want to select rows where a column matches **any one of multiple values**, using `.isin()` is the best approach. <br>
Instead of chaining multiple `|` (OR) conditions, you can simply pass a list of allowed values.<br><br>
&nbsp;&nbsp;&nbsp;&nbsp;df[ df[Column].<b>isin</b>( [<i>list of conditions</i>] ) ]

e.g.> rows where `Department` is "Data", "Cloud", or "Security"

In [25]:
selected_department = df_students[
    df_students["Department"].isin(["Data","Cloud","Security"])
]

selected_department

# Above is better than below:
# (df_students["Department"] == "Data") | (df_students["Department"] == "Cloud") | (df_students["Department"] == "Security")

,Name,Age,Score,Department,Passed
0,Alice,25,88,Data,True
2,Charlie,35,79,Data,False
3,David,28,85,Cloud,True
5,Frank,40,95,Security,True


## 09. Selecting a Range of Numbers: `between()`

When you want to select rows where a numeric value falls **within a specific range**, you can use `.between()`.

This method is inclusive by default, meaning it includes both the start and end values (equivalent to `>=` and `<=`).

```python
# ❌ The traditional way (Using two comparison operators)
df[(df["Score"] >= 80) & (df["Score"] <= 90)]

# 🟢 The cleaner way (Using .between())
df[df["Score"].between(80, 90)]

In [27]:
score_between = df_students[ df_students["Score"].between(80, 90) ]
score_between

,Name,Age,Score,Department,Passed
0,Alice,25,88,Data,True
3,David,28,85,Cloud,True


## 10. String Filtering
You can also create filtering conditions for **string (text) columns**. 
By using the `.str` accessor, you can easily filter rows based on partial matches, specific prefixes, or suffixes.


- .str.**contains**( ): Select rows where the text contains a specific substring
```python
df[df["Name"].str.contains("A")]
```

- .str.**startswith**( ): Select rows where the text starts with a specific string
```python
df[df["Email"].str.startswith("admin")]
```

- .str.**endswith**( ): Select rows where the text ends with a specific string
```python
df[df["Email"].str.endswith(".com")]
````

In [75]:
# Filter rows where Department contains "Data"
contains_data = df_students[ df_students["Department"].str.contains("Data") ]
contains_data

,Name,Age,Score,Department,Passed
0,Alice,25,88,Data,True
2,Charlie,35,79,Data,False


In [32]:
# Filter rows where Name starts with A
name_starts_with = df_students[ df_students["Name"].str.startswith("A") ]
name_starts_with

,Name,Age,Score,Department,Passed
0,Alice,25,88,Data,True


In [33]:
# Filter rows where Name ends with a
name_ends_with = df_students[ df_students["Name"].str.endswith("a") ]
name_ends_with

,Name,Age,Score,Department,Passed
4,Eva,22,67,Engineering,False


## 11. Filtering with `loc[]`

Using `.loc[]` is the most powerful and preferred way to filter data in pandas. 

The biggest advantage of `.loc[]` is that it allows you to **filter rows and select specific columns at the same time** within a single bracket.

#### 💡 Syntax Structure
```python
df.loc[Row_Condition, List_of_Column_Selection]

In [34]:
# Filter rows using loc
high_score_students = df_students.loc[df_students["Score"] >= 85]

high_score_students

,Name,Age,Score,Department,Passed
0,Alice,25,88,Data,True
1,Bob,30,92,Engineering,True
3,David,28,85,Cloud,True
5,Frank,40,95,Security,True


In [37]:
# filter rows and select specific columns at the same time
high_score_name_score = df_students.loc[
    df_students["Score"] >= 85, ["Name", "Score"]
]

high_score_name_score

,Name,Score
0,Alice,88
1,Bob,92
3,David,85
5,Frank,95


## Practice Problems

### Practice 1 — Create a Sample CSV File
#### Goal: Practice creating a CSV file for row filtering practice.
##### <u>Requirements</u>
&nbsp;&nbsp;&nbsp;&nbsp;Create a DataFrame with the following columns:<br>
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;Name<br>
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;Age<br>
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;Score<br>
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;Department<br>
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;Passed<br><br>
&nbsp;&nbsp;&nbsp;&nbsp;Use at least six rows of data.<br>
&nbsp;&nbsp;&nbsp;&nbsp;Save it as:<br>
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;../assets/Week05/Day04/Output/students.csv

In [76]:
path_dir = "../assets/Week05/Day04/Output/"
file_name = "students.csv"
full_file_path = f"{path_dir}{file_name}"

# Create input folder if it doesn't exist
os.makedirs(path_dir, exist_ok=True)

# Create sample student data
student_data = {
    "Name": ["Alice", "Bob", "Charlie", "David", "Eva", "Frank"],
    "Age": [25, 30, 35, 28, 22, 40],
    "Score": [88, 92, 79, 85, 67, 95],
    "Department": ["Data", "Engineering", "Data", "Cloud", "Engineering", "Security"],
    "Passed": [True, True, False, True, False, True]
}

pr_df_students = pd.DataFrame(student_data)

pr_df_students.to_csv(full_file_path)

print(pr_df_students)

      Name  Age  Score   Department  Passed
0    Alice   25     88         Data    True
1      Bob   30     92  Engineering    True
2  Charlie   35     79         Data   False
3    David   28     85        Cloud    True
4      Eva   22     67  Engineering   False
5    Frank   40     95     Security    True


### Practice 2 — Read the CSV File
#### Goal: Practice reading the CSV file before filtering rows.
##### <u>Requirements</u>
&nbsp;&nbsp;&nbsp;&nbsp;Read the following CSV file:<br>
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;../assets/Week05/Day04/Input/students.csv<br>
&nbsp;&nbsp;&nbsp;&nbsp;Store it in a variable called df_students.<br>
&nbsp;&nbsp;&nbsp;&nbsp;Display the DataFrame.

In [45]:
import os
import pandas as pd

path_dir = "../assets/Week05/Day04/Input/"
file_name = "students.csv"
full_file_path = f"{path_dir}{file_name}"

if os.path.exists(full_file_path):
    df_students = pd.read_csv(full_file_path)
else:
    student_data = {
        "Name": ["Alice", "Bob", "Charlie", "David", "Eva", "Frank"],
        "Age": [25, 30, 35, 28, 22, 40],
        "Score": [88, 92, 79, 85, 67, 95],
        "Department": ["Data", "Engineering", "Data", "Cloud", "Engineering", "Security"],
        "Passed": [True, True, False, True, False, True]
    }
    df_students = pd.DataFrame(student_data)
    df_students.to_csv(full_file_path)


print(df_students)

      Name  Age  Score   Department  Passed
0    Alice   25     88         Data    True
1      Bob   30     92  Engineering    True
2  Charlie   35     79         Data   False
3    David   28     85        Cloud    True
4      Eva   22     67  Engineering   False
5    Frank   40     95     Security    True


### Practice 3 — Filter High Score Students
#### Goal: Practice filtering rows using one numeric condition.
##### <u>Requirements</u>
&nbsp;&nbsp;&nbsp;&nbsp;Select students whose Score is greater than or equal to 85.<br>
&nbsp;&nbsp;&nbsp;&nbsp;Store the result in high_score_students.

In [51]:
high_score_students = df_students[ df_students["Score"]>=85 ]

print(high_score_students)

    Name  Age  Score   Department  Passed
0  Alice   25     88         Data    True
1    Bob   30     92  Engineering    True
3  David   28     85        Cloud    True
5  Frank   40     95     Security    True


### Practice 4 — Filter Students by Department
#### Goal: Practice filtering rows using a string condition.
##### <u>Requirements</u>
&nbsp;&nbsp;&nbsp;&nbsp;Select students whose Department is "Data".<br>
&nbsp;&nbsp;&nbsp;&nbsp;Store the result in data_department_students.

In [50]:
data_department_students = df_students[ df_students["Department"]=="Data" ]

print(data_department_students)

      Name  Age  Score Department  Passed
0    Alice   25     88       Data    True
2  Charlie   35     79       Data   False


### Practice 5 — Filter Passed Students
#### Goal: Practice filtering rows using a boolean column.
##### <u>Requirements</u>
&nbsp;&nbsp;&nbsp;&nbsp;Select students whose Passed value is True.<br>
&nbsp;&nbsp;&nbsp;&nbsp;Store the result in passed_students.

In [49]:
passed_students = df_students[ df_students["Passed"] ]

print(passed_students)

    Name  Age  Score   Department  Passed
0  Alice   25     88         Data    True
1    Bob   30     92  Engineering    True
3  David   28     85        Cloud    True
5  Frank   40     95     Security    True


### Practice 6 — Filter with Multiple Conditions
#### Goal: Practice filtering rows using multiple conditions.
##### <u>Requirements</u>
&nbsp;&nbsp;&nbsp;&nbsp;Select students whose:<br>
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;Score is greater than or equal to 85<br>
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;AND<br>
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;Age is greater than or equal to 30<br><br>
&nbsp;&nbsp;&nbsp;&nbsp;Store the result in filtered_students.

In [74]:
filtered_students = df_students[ (df_students["Score"]>=30) & (df_students["Score"]<=85) ]
print(f"Using &:\n {filtered_students}\n")

Using &:
       Name  Age  Score   Department  Passed
2  Charlie   35     79         Data   False
3    David   28     85        Cloud    True
4      Eva   22     67  Engineering   False



### Practice 7 — Filter with OR Condition
#### Goal: Practice filtering rows using the OR condition.
##### <u>Requirements</u>
&nbsp;&nbsp;&nbsp;&nbsp;Select students whose Department is either "Data" or "Cloud".<br>
&nbsp;&nbsp;&nbsp;&nbsp;Store the result in data_or_cloud_students.

In [67]:
data_or_cloud_students = df_students[
        (df_students["Department"]=="Data") | (df_students["Department"]=="Cloud")
    ]

print(f"Using | :\n{data_or_cloud_students}\n")

data_or_cloud_students = df_students[
        df_students["Department"].isin(["Data", "Cloud"])
    ]

print(f"Using .isin() :\n{data_or_cloud_students}")

Using | :
      Name  Age  Score Department  Passed
0    Alice   25     88       Data    True
2  Charlie   35     79       Data   False
3    David   28     85      Cloud    True

Using .isin() :
      Name  Age  Score Department  Passed
0    Alice   25     88       Data    True
2  Charlie   35     79       Data   False
3    David   28     85      Cloud    True


### Practice 8 — Filter with isin()
#### Goal: Practice filtering rows using isin().
##### <u>Requirements</u>
&nbsp;&nbsp;&nbsp;&nbsp;Select students whose Department is one of the following:<br>
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;Data<br>
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;Cloud<br>
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;Security<br><br>
&nbsp;&nbsp;&nbsp;&nbsp;Store the result in selected_department_students.<br>

In [70]:
selected_department_students = df_students[
        df_students["Department"].isin(["Data", "Cloud", "Security"])
    ]

print(selected_department_students)

      Name  Age  Score Department  Passed
0    Alice   25     88       Data    True
2  Charlie   35     79       Data   False
3    David   28     85      Cloud    True
5    Frank   40     95   Security    True


### Practice 9 — Filter with between()
#### Goal: Practice filtering rows using between().
##### <u>Requirements</u>
&nbsp;&nbsp;&nbsp;&nbsp;Select students whose Score is between 80 and 90.<br>
&nbsp;&nbsp;&nbsp;&nbsp;Store the result in score_between_80_90.

In [71]:
score_between_80_90 = df_students[
        df_students["Score"].between(80, 90)
    ]

print(score_between_80_90)

    Name  Age  Score Department  Passed
0  Alice   25     88       Data    True
3  David   28     85      Cloud    True


### Practice 10 — Filter and Select Columns with loc[]
#### Goal: Practice filtering rows and selecting columns at the same time.
##### <u>Requirements</u>
&nbsp;&nbsp;&nbsp;&nbsp;Select students whose Score is greater than or equal to 85.<br><br>
&nbsp;&nbsp;&nbsp;&nbsp;Return only the following columns:<br>
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;Name<br>
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;Score<br>
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;Department<br><br>
&nbsp;&nbsp;&nbsp;&nbsp;Store the result in high_score_selected_columns.

In [73]:
high_score_selected_columns = df_students.loc[
        df_students["Score"]>=85, ["Name", "Score", "Department"]
    ]

print(high_score_selected_columns)

    Name  Score   Department
0  Alice     88         Data
1    Bob     92  Engineering
3  David     85        Cloud
5  Frank     95     Security


## Summary

Today, I practiced row filtering in pandas.

I learned how to filter rows using boolean conditions such as numeric comparisons, string comparisons, and boolean columns.

I also practiced filtering with multiple conditions using `&` and `|`, and learned that each condition should be wrapped in parentheses.

I used `isin()` to filter rows based on multiple possible values and `between()` to filter numeric values within a range.

Finally, I practiced using `.loc[]` to filter rows and select specific columns at the same time.

These concepts are important because row filtering is one of the most common operations in data analysis, machine learning, and data engineering workflows.